# Critical Ising Entanglement Scan (D=8)

This notebook plots the mutual information and relative entropy data from `ising_entanglement_scan_D8.csv` for the pairwise code subspaces `(I, sigma)`, `(I, epsilon)`, and `(sigma, epsilon)`.

In [ ]:
from pathlib import Path
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")
import csv
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 20,
    "axes.labelsize": 24,
    "axes.titlesize": 26,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 18,
    "lines.linewidth": 6,
    "lines.markersize": 50,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "axes.linewidth": 1.5,
    "grid.linestyle": "--",
    "grid.linewidth": 2.5,
})

base_dir = Path.cwd()
if not (base_dir / "ising_entanglement_scan_D8.csv").exists():
    base_dir = Path.cwd() / "examples"

csv_path = base_dir / "ising_entanglement_scan_D8.csv"
fig_dir = base_dir
print("reading:", csv_path)

In [ ]:
rows = []
with csv_path.open() as f:
    reader = csv.DictReader(f)
    for r in reader:
        rows.append({
            "N": int(r["N"]),
            "ell": int(r["ell"]),
            "x": float(r["x"]),
            "I01": float(r["I01"]),
            "I02": float(r["I02"]),
            "I12": float(r["I12"]),
            "S10": float(r["S10"]),
            "S20": float(r["S20"]),
            "S12": float(r["S12"]),
        })

print(f"loaded {len(rows)} rows")
print("L range:", min(r["N"] for r in rows), "to", max(r["N"] for r in rows))
print("ell values:", sorted(set(r["ell"] for r in rows)))

In [ ]:
marker_cycle = ["o", "s", "^", "D", "v", "P", "X"]

pair_specs = [
    ("I01", r"$I$--$\sigma$", "tab:blue", marker_cycle[0]),
    ("I02", r"$I$--$\epsilon$", "tab:orange", marker_cycle[1]),
    ("I12", r"$\sigma$--$\epsilon$", "tab:green", marker_cycle[2]),
]

rel_specs = [
    ("S10", r"$S(\rho_\sigma\Vert\rho_I)$", "tab:blue", marker_cycle[0]),
    ("S20", r"$S(\rho_\epsilon\Vert\rho_I)$", "tab:orange", marker_cycle[1]),
    ("S12", r"$S(\rho_\sigma\Vert\rho_\epsilon)$", "tab:green", marker_cycle[2]),
]

def fit_power_law(xs, ys):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    mask = (xs > 0) & (ys > 0)
    logx = np.log(xs[mask])
    logy = np.log(ys[mask])
    beta = np.linalg.lstsq(np.column_stack([np.ones_like(logx), logx]), logy, rcond=None)[0]
    return math.exp(beta[0]), beta[1]

def plot_observable(specs, ylabel, fig_stem, *, fit_max_x=0.5):
    fig, ax = plt.subplots(1, 1, figsize=(20, 15))
    fit_summaries = []

    for key, label, color, marker in specs:
        data = [{**r, "y": r[key], "used_in_fit": (r["x"] < fit_max_x and r[key] > 0)} for r in rows]
        used = [r for r in data if r["used_in_fit"]]
        excluded = [r for r in data if not r["used_in_fit"]]

        x_used = np.array([r["x"] for r in used], dtype=float)
        y_used = np.array([r["y"] for r in used], dtype=float)
        sort_idx = np.argsort(x_used)
        x_used = x_used[sort_idx]
        y_used = y_used[sort_idx]

        ax.plot(x_used, y_used, linestyle="None", marker=marker, color=color,
                label=label, markersize=28)

        if excluded:
            x_ex = np.array([r["x"] for r in excluded], dtype=float)
            y_ex = np.array([r["y"] for r in excluded], dtype=float)
            ax.plot(x_ex, y_ex, linestyle="None", marker="x", color=color,
                    alpha=0.35, markersize=24)

        A, gamma = fit_power_law(x_used, y_used)
        fit_summaries.append((label, A, gamma))
        x_line = np.logspace(np.log10(x_used.min() * 0.95), np.log10(x_used.max() * 1.05), 300)
        y_line = A * (x_line ** gamma)
        ax.plot(x_line, y_line, linestyle="--", alpha=0.8, color=color)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$x = \ell / L$", fontsize=72)
    ax.set_ylabel(ylabel, fontsize=72)
    ax.tick_params(axis="both", labelsize=60)
    ax.set_xticks([0.08, 0.1, 0.2, 0.3, 0.4, 0.5])
    ax.set_xticklabels([r"$0.08$", r"$0.1$", r"$0.2$", r"$0.3$", r"$0.4$", r"$0.5$"], fontsize=50)
    ax.grid(True, which="both")
    ax.legend(loc="upper left", fontsize=42)

    text = "\n".join([rf"{label}: $\gamma={gamma:.4f}$, $A={A:.4f}$" for label, A, gamma in fit_summaries])
    ax.text(0.33, 0.08, text, transform=ax.transAxes, fontsize=38,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.8))

    fig.tight_layout()
    fig_pdf_path = fig_dir / f"{fig_stem}.pdf"
    fig_png_path = fig_dir / f"{fig_stem}.png"
    fig.savefig(fig_pdf_path, dpi=300, bbox_inches="tight")
    fig.savefig(fig_png_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("saved figure pdf:", fig_pdf_path)
    print("saved figure png:", fig_png_path)
    return fit_summaries

In [ ]:
mi_fits = plot_observable(
    pair_specs,
    r"$I(Q_1:R)$ (bits)",
    "ising_mutual_information_D8",
    fit_max_x=0.5,
)
mi_fits

In [ ]:
rel_fits = plot_observable(
    rel_specs,
    r"$S(\rho_A\Vert\rho_B)$",
    "ising_relative_entropy_D8",
    fit_max_x=0.5,
)
rel_fits